In [19]:
import torch
import torchaudio
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [20]:
import torch
import torchaudio
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [21]:
def load_audio(path):
    wav, sr = torchaudio.load(path)

    # mono
    wav = wav.mean(dim=0)

    # resample
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)

    # pad / truncate
    if wav.shape[0] < MAX_LEN:
        wav = torch.nn.functional.pad(wav, (0, MAX_LEN - wav.shape[0]))
    else:
        wav = wav[:MAX_LEN]

    return wav


In [22]:
emotion_to_id = {
    "neutral": 0,
    "happy": 1,
    "sad": 2,
    "angry": 3,
    "fear": 4
}

domain_to_id = {
    "ravdess": 0,
    "crema_d": 1
}


In [23]:
class SERDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        wav = load_audio(row["path"])
        emotion = emotion_to_id[row["emotion"]]
        domain = domain_to_id[row["dataset"]]

        return wav, emotion, domain


In [24]:
train_ds = SERDataset(r"D:\SER_Cross\data\processed\train.csv")
val_ds   = SERDataset(r"D:\SER_Cross\data\processed\val.csv")
test_ds  = SERDataset(r"D:\SER_Cross\data\processed\test.csv")

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=4, shuffle=False)

print(len(train_ds), len(val_ds), len(test_ds))


5068 1031 1128


In [25]:
wav, emo, dom = next(iter(train_loader))

print("Waveform shape:", wav.shape)
print("Emotion labels:", emo)
print("Domain labels:", dom)


Waveform shape: torch.Size([4, 96000])
Emotion labels: tensor([2, 3, 3, 3])
Domain labels: tensor([1, 1, 1, 1])


In [26]:
from transformers import Wav2Vec2Model

wav2vec = Wav2Vec2Model.from_pretrained(
    "facebook/wav2vec2-xls-r-300m"
).to(device)

print("Wav2Vec2 loaded")


c:\Users\Asus\anaconda3\envs\wav2vec_ser\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wav2Vec2 loaded


In [27]:
for param in wav2vec.parameters():
    param.requires_grad = False

print("Wav2Vec2 frozen")


Wav2Vec2 frozen


In [28]:
class Wav2VecEmotion(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(1024, 5)

    def forward(self, wav):
        out = self.encoder(wav).last_hidden_state   # [B, T', 1024]
        out = out.transpose(1, 2)                   # [B, 1024, T']
        out = self.pool(out).squeeze(-1)            # [B, 1024]
        return self.classifier(out)


In [29]:
model = Wav2VecEmotion(wav2vec).to(device)


In [30]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.classifier.parameters(),  # only head
    lr=1e-3
)


In [31]:
def train_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for wav, emo, _ in tqdm(loader):
        wav = wav.to(device)
        emo = emo.to(device)

        optimizer.zero_grad()
        logits = model(wav)
        loss = criterion(logits, emo)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == emo).sum().item()
        total += emo.size(0)

    return total_loss / len(loader), correct / total


In [ ]:
for epoch in range(3):
    loss, acc = train_epoch(model, train_loader)
    print(f"Epoch {epoch+1}: loss={loss:.4f}, train_acc={acc:.4f}")


 61%|██████    | 770/1267 [1:55:56<1:14:50,  9.03s/it]


KeyboardInterrupt: 

: 

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for wav, emo, _ in loader:
            wav = wav.to(device)
            emo = emo.to(device)

            logits = model(wav)
            preds = logits.argmax(dim=1)
            correct += (preds == emo).sum().item()
            total += emo.size(0)

    return correct / total


In [ ]:
val_acc = evaluate(model, val_loader)
print("Baseline Validation Accuracy:", val_acc)
